# Sudoku-09 : Coloration de Graphe (Rust)

**Navigation** : [<< Human Stratégies](Sudoku-08-HumanStrategies-Python.ipynb) | [Index](README.md) | [OR-Tools >>](Sudoku-10-ORTools-Python.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Modéliser** un Sudoku comme un problème de coloration de graphe
2. **Comprendre** la théorie des graphes sous-jacente (81 sommets, degré 20)
3. **Utiliser** la crate `petgraph` pour manipuler des graphes de contraintes
4. **Comparer** l'efficacité de différentes stratégies de coloration (MRV, LCV, Welsh-Powell)

**Durée estimée** : ~15 min | **Prerequis** : Sudoku-00 Environment | **Jumeaux** : [Python](Sudoku-09-GraphColoring-Python.ipynb) · [C#](Sudoku-09-GraphColoring-CSharp.ipynb)

***

Ce carnet est le jumeau Rust de **Sudoku-09 : Coloration de Graphe**. Il modélise le Sudoku comme un problème de coloration de graphe et résout la grille par backtracking guidé par le graphe, avec la crate **`petgraph`** comme organe de structure de graphe, exécutée en place par le kernel **evcxr**.

La théorie complète (coloration propre, heuristiques MRV / LCV / Welsh-Powell) vit dans le jumeau Python ; ici, l'accent est mis sur **l'organe natif Rust** et ce que son API change au portage : `networkx` livre un générateur `sudoku_graph()` prêt à l'emploi, `petgraph` livre la **structure** et laisse la construction du graphe au carnet — c'est une cellule de cours à part entière.

## Table des matières

1. [L'organe : la crate petgraph](#1-lorgane-la-crate-petgraph)
2. [Le graphe Sudoku : 81 sommets, 810 aretes](#2-le-graphe-sudoku-81-sommets-810-aretes)
3. [Solveurs : backtracking guidé par le graphe](#3-solveurs-backtracking-guide-par-le-graphe)
4. [Exercices](#4-exercices)

## References

- [Crate petgraph (crates.io)](https://crates.io/crates/petgraph) — licence MIT / Apache-2.0
- [NetworkX `sudoku_graph` (doc du jumeau Python)](https://networkx.org/documentation/stable/reference/generated/networkx.generators.sudoku.sudoku_graph.html)
- [evcxr — le kernel Jupyter pour Rust](https://github.com/evcxr/evcxr)

## 1. L'organe : la crate petgraph

Le jumeau Python s'appuie sur **`networkx`**, qui fournit `nx.sudoku_graph()` : le graphe de contraintes du Sudoku en un appel. En Rust, l'organe natif de manipulation de graphes est la crate **`petgraph`** (MIT / Apache-2.0) — mais elle ne fournit **pas** de générateur de Sudoku : elle livre la structure (`Graph`, `neighbors`, `node_count`, `edge_count`), et la construction des 810 aretes est notre travail. C'est la première leçon du portage : l'organe Rust est plus fin-grained que le générateur Python.

| | Jumeau Python | Ce carnet (Rust) |
|---|---|---|
| Graphe de contraintes | `nx.sudoku_graph()` (générateur intégré) | **crate `petgraph`** + construction explicite |
| Voisinage | `G.neighbors(v)` | `graph.neighbors(NodeIndex)` |
| Statistiques | `G.number_of_nodes()` / `G.number_of_edges()` | `graph.node_count()` / `graph.edge_count()` |
| Solveur | backtracking MRV maison sur le graphe | idem — backtracking MRV maison sur `petgraph` |

La cellule `:dep` ci-dessous ajoute la dépendance cargo **directement dans le kernel** — l'équivalent Rust des cellules `# pip install` (Python) ou `#r "nuget:"` (.NET). La règle du carnet : `:dep` **en tête uniquement** — un `:dep` ajouté plus bas recompilerait tout et effacerait les variables typées déjà définies.

In [2]:
:dep petgraph = "0.8"

## 2. Le graphe Sudoku : 81 sommets, 810 aretes

### Modélisation

- **Sommets** : les 81 cellules de la grille (index `v = ligne * 9 + colonne`)
- **Aretes** : deux cellules sont reliées si elles ne peuvent pas porter la même valeur (même ligne, même colonne, ou même bloc 3x3)
- **Couleurs** : les valeurs 1 à 9 — résoudre le Sudoku = **colorer proprement** le graphe

### Propriétés attendues

- **81 sommets**, **810 aretes** (chaque couple en conflit compté une fois)
- **Graphe régulier de degré 20** : chaque cellule a 8 voisins de ligne + 8 de colonne + 4 de bloc (hors ligne et colonne)

### Construction avec petgraph

`petgraph::Graph` accepte les aretes parallèles : pour tomber à 810 aretes exactement, on ne relie chaque couple `(v, w)` qu'une seule fois, en itérant `w > v`. Les 81 sommets sont ajoutés d'abord : l'indice du noeud `NodeIndex` coïncide alors avec l'index de cellule par construction, et `neighbors(NodeIndex::new(v))` livre le voisinage de la cellule `v`.

Sous evcxr, les fonctions ne persistent pas fiablement d'une cellule à la suivante : chaque cellule de ce carnet est donc **autonome** — elle enveloppe ses définitions dans un bloc `{ ... }` et les redéfinit localement.

In [3]:
// Construction du graphe Sudoku avec petgraph et verification des proprietes
{
    use petgraph::graph::NodeIndex;
    use petgraph::{Graph, Undirected};

    fn sudoku_graph() -> Graph<(), (), Undirected> {
        // 81 sommets (un par cellule), puis une arete par couple en conflit (w > v)
        let mut g: Graph<(), (), Undirected> = Graph::with_capacity(81, 810);
        for _ in 0..81 {
            g.add_node(());
        }
        for v in 0..81usize {
            let (r, c) = (v / 9, v % 9);
            for w in (v + 1)..81usize {
                let (r2, c2) = (w / 9, w % 9);
                let meme_ligne = r == r2;
                let meme_colonne = c == c2;
                let meme_bloc = r / 3 == r2 / 3 && c / 3 == c2 / 3;
                if meme_ligne || meme_colonne || meme_bloc {
                    g.add_edge(NodeIndex::new(v), NodeIndex::new(w), ());
                }
            }
        }
        g
    }

    let g = sudoku_graph();
    let degres: Vec<usize> = g.node_indices().map(|v| g.neighbors(v).count()).collect();
    let deg_min = degres.iter().copied().min();
    let deg_max = degres.iter().copied().max();
    let regulier = deg_min == deg_max;
    println!("=== Statistiques du Graphe Sudoku ===");
    println!("Sommets: {}", g.node_count());
    println!("Aretes: {}", g.edge_count());
    println!("Degré min: {}, max: {}", deg_min.unwrap_or(0), deg_max.unwrap_or(0));
    println!("Graphe regulier: {}", regulier);
};

=== Statistiques du Graphe Sudoku ===


Sommets: 81


Aretes: 810


Degré min: 20, max: 20


Graphe regulier: true


### Interprétation

`petgraph` confirme la structure attendue :

- **81 sommets** (index 0-80), **810 aretes** — le même compte que `nx.sudoku_graph()` du jumeau Python
- **Degré min = max = 20** : le graphe est **régulier**, chaque cellule a exactement 20 voisins en conflit (8 de ligne, 8 de colonne, 4 de bloc)

C'est cette régularité qui rend le problème bien posé pour la coloration — et qui, on le verra en section 3, rend certaines heuristiques de tri par degré **inopérantes** : tous les degrés sont identiques.

### Grille et coloration : la même structure qu'en Python

Le jumeau Python enveloppe la grille dans une classe `SudokuGrid` ; ici, la coloration est directement un `Vec<usize>` de 81 cases (`0` = non colorié, `1..=9` = couleur). Les instances de test sont celles du jumeau Python, reprises telles quelles depuis `Puzzles/Sudoku_Easy51.txt` (faciles) et `Puzzles/Sudoku_hardest.txt` (difficiles) — le carnet Rust les inline pour rester autonome et indépendant du répertoire de travail du kernel.

Le format d'entrée accepte indifféremment `0` (fichiers faciles) et `.` (fichiers difficiles, format du jumeau Python) comme case vide.

In [4]:
// Conversion grille <-> coloration, et affichage d'une instance
{
    fn parse_coloring(s: &str) -> Vec<usize> {
        // Chiffre '1'-'9' -> valeur ; '0' ou '.' -> case vide
        s.chars()
            .filter(|ch| ch.is_ascii_digit() || *ch == '.')
            .map(|ch| if ch == '.' { 0 } else { ch as usize - '0' as usize })
            .collect()
    }

    fn print_grid(coloring: &[usize]) {
        for r in 0..9usize {
            if r > 0 && r % 3 == 0 {
                println!("{}", "-".repeat(21));
            }
            let mut ligne = String::new();
            for c in 0..9usize {
                if c > 0 && c % 3 == 0 {
                    ligne.push_str("| ");
                }
                let v = coloring[r * 9 + c];
                if v == 0 {
                    ligne.push_str(". ");
                } else {
                    ligne.push_str(&format!("{} ", v));
                }
            }
            println!("{}", ligne);
        }
    }

    // easy[0] -- Puzzles/Sudoku_Easy51.txt, ligne 1 (0 = case vide)
    let facile0 = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    println!("Puzzle facile (Puzzles/Sudoku_Easy51.txt, ligne 1) :");
    print_grid(&parse_coloring(facile0));
    let indices = parse_coloring(facile0).iter().filter(|&&v| v != 0).count();
    println!("\nIndices (cases pre-remplies) : {}", indices);
};

Puzzle facile (Puzzles/Sudoku_Easy51.txt, ligne 1) :


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


---------------------


. 2 6 | 3 . 9 | . . 1 


... (3 outputs left out)

2 4 . | 5 3 . | 6 . . 


7 . 5 | 2 . . | 3 . 4 


. 8 . | . 4 1 | 9 5 . 


Indices (cases pre-remplies) : 45


## 3. Solveurs : backtracking guidé par le graphe

### MRV — Minimum Remaining Values

L'heuristique **MRV** choisit à chaque étape la case non coloriée au **domaine le plus petit** (le moins de couleurs encore disponibles au vu de ses 20 voisins) : les cases les plus contraintes sont résolues d'abord, et l'arbre de recherche reste étroit.

Le solveur parcourt le graphe via des **listes d'adjacence** précalculées depuis `petgraph` (`graph.neighbors(NodeIndex)`, projetées en `Vec<Vec<usize>>`), puis effectue un backtracking classique :

1. sélectionner le sommet non colorié avec le moins de couleurs disponibles (première occurrence en cas d'égalité) ;
2. essayer chaque couleur disponible **par ordre croissant** ;
3. en cas d'impasse, décolorer et remonter (**backtrack**).

Deux compteurs (`noeuds`, `retours`) mesurent la taille de l'exploration — ce sont les grandeurs que la comparaison d'heuristiques de la fin de section fera parler.

In [5]:
// Solveur MRV par backtracking sur le graphe de contraintes
{
    use petgraph::graph::NodeIndex;
    use petgraph::{Graph, Undirected};

    fn sudoku_graph() -> Graph<(), (), Undirected> {
        let mut g: Graph<(), (), Undirected> = Graph::with_capacity(81, 810);
        for _ in 0..81 {
            g.add_node(());
        }
        for v in 0..81usize {
            let (r, c) = (v / 9, v % 9);
            for w in (v + 1)..81usize {
                let (r2, c2) = (w / 9, w % 9);
                let conflit = r == r2 || c == c2 || (r / 3 == r2 / 3 && c / 3 == c2 / 3);
                if conflit {
                    g.add_edge(NodeIndex::new(v), NodeIndex::new(w), ());
                }
            }
        }
        g
    }

    fn adjacence(g: &Graph<(), (), Undirected>) -> Vec<Vec<usize>> {
        (0..81usize)
            .map(|v| g.neighbors(NodeIndex::new(v)).map(|n| n.index()).collect())
            .collect()
    }

    fn parse_coloring(s: &str) -> Vec<usize> {
        s.chars()
            .filter(|ch| ch.is_ascii_digit() || *ch == '.')
            .map(|ch| if ch == '.' { 0 } else { ch as usize - '0' as usize })
            .collect()
    }

    fn print_grid(coloring: &[usize]) {
        for r in 0..9usize {
            if r > 0 && r % 3 == 0 {
                println!("{}", "-".repeat(21));
            }
            let mut ligne = String::new();
            for c in 0..9usize {
                if c > 0 && c % 3 == 0 {
                    ligne.push_str("| ");
                }
                let v = coloring[r * 9 + c];
                if v == 0 {
                    ligne.push_str(". ");
                } else {
                    ligne.push_str(&format!("{} ", v));
                }
            }
            println!("{}", ligne);
        }
    }

    fn couleurs_disponibles(v: usize, coloring: &[usize], adj: &[Vec<usize>]) -> Vec<usize> {
        let mut pris = [false; 10];
        for &n in &adj[v] {
            pris[coloring[n]] = true;
        }
        (1..=9usize).filter(|&d| !pris[d]).collect()
    }

    fn select_mrv(coloring: &[usize], adj: &[Vec<usize>]) -> Option<usize> {
        // Sommet non colorie au domaine minimal ; premiere occurrence en cas d'egalite
        let mut meilleur: Option<usize> = None;
        let mut min_couleurs = usize::MAX;
        for v in 0..81usize {
            if coloring[v] != 0 {
                continue;
            }
            let n = couleurs_disponibles(v, coloring, adj).len();
            if n < min_couleurs {
                min_couleurs = n;
                meilleur = Some(v);
            }
        }
        meilleur
    }

    fn backtrack_mrv(
        coloring: &mut [usize],
        adj: &[Vec<usize>],
        noeuds: &mut usize,
        retours: &mut usize,
    ) -> bool {
        *noeuds += 1;
        let sommet = match select_mrv(coloring, adj) {
            None => return true, // toutes les cases sont coloriees
            Some(v) => v,
        };
        for couleur in couleurs_disponibles(sommet, coloring, adj) {
            coloring[sommet] = couleur;
            if backtrack_mrv(coloring, adj, noeuds, retours) {
                return true;
            }
            coloring[sommet] = 0;
            *retours += 1;
        }
        false
    }

    fn resoudre_mrv(puzzle: &str, adj: &[Vec<usize>]) -> (bool, usize, usize, Vec<usize>) {
        let mut coloring = parse_coloring(puzzle);
        let mut noeuds = 0;
        let mut retours = 0;
        let ok = backtrack_mrv(&mut coloring, adj, &mut noeuds, &mut retours);
        (ok, noeuds, retours, coloring)
    }

    // Resolution de easy[0]
    let g = sudoku_graph();
    let adj = adjacence(&g);
    let facile0 = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    let t0 = std::time::Instant::now();
    let (succes, noeuds, retours, solution) = resoudre_mrv(facile0, &adj);
    let ms = t0.elapsed().as_secs_f64() * 1000.0;
    println!("\nResolu: {} en {:.2} ms", succes, ms);
    println!("Noeuds explores: {}", noeuds);
    println!("Backtracks: {}", retours);
    println!("\nSolution :");
    print_grid(&solution);
};

Resolu: true en 0.07 ms


Noeuds explores: 37


Backtracks: 0


Solution :


9 6 2 | 1 8 5 | 4 7 3 


1 7 4 | 9 6 3 | 8 2 5 


... (5 outputs left out)

---------------------


2 4 9 | 5 3 8 | 6 1 7 


7 1 5 | 2 9 6 | 3 8 4 


6 8 3 | 7 4 1 | 9 5 2 


### Interprétation : backtracking avec MRV sur le graphe

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| **Noeuds explorés** | 37 | 36 cases à remplir + l'appel terminal qui constate la grille complète |
| **Backtracks** | 0 | aucune impasse : chaque première couleur essayée est la bonne |
| **Temps** | 0.07 ms | résolution pure — le jumeau Python mesure ~2 ms sur la même instance |

**Points clés** :

1. **MRV fonctionne très bien** : l'heuristique choisit d'abord les cases les plus contraintes, et la grille facile (45 indices) tombe sans un seul retour arrière. Le comptage 37 = 36 + 1 est **indépendant de l'ordre d'essai des couleurs** : sans impasse, il n'y a rien à départager.
2. **`petgraph` porte la structure, pas la stratégie** : le voisinage vient du graphe (`adj[v]`), le choix MRV reste notre code — `networkx` non plus ne fournit pas de solveur intégré.
3. **Graphe régulier** : tous les sommets ont degré 20, donc MRV se joue entièrement sur les **domaines**, pas sur les degrés.

### Benchmark comparatif

Les mêmes instances que le jumeau Python : **3 puzzles faciles** (`Sudoku_Easy51.txt`, lignes 1-3) et **2 puzzles réputés difficiles** (`Sudoku_hardest.txt`, lignes 1-2). Les temps mesurés sont ceux de la **résolution seule** — le chrono `Instant::now` ne compte ni la compilation evcxr de la cellule, ni la construction du graphe.

Une divergence assumée du portage : le jumeau Python itère les couleurs candidates dans un `set` (ordre interne de CPython), ce carnet itère **par ordre croissant**. Sur les puzzles qui forcent des backtracks, les comptages `noeuds`/`backtracks` peuvent donc différer du jumeau — la recherche n'explore pas les mêmes branches. La comparaison **entre heuristiques, à ordre constant**, reste le propos du carnet.

In [6]:
// Benchmark du solveur MRV sur les instances du jumeau Python
{
    use petgraph::graph::NodeIndex;
    use petgraph::{Graph, Undirected};

    fn sudoku_graph() -> Graph<(), (), Undirected> {
        let mut g: Graph<(), (), Undirected> = Graph::with_capacity(81, 810);
        for _ in 0..81 {
            g.add_node(());
        }
        for v in 0..81usize {
            let (r, c) = (v / 9, v % 9);
            for w in (v + 1)..81usize {
                let (r2, c2) = (w / 9, w % 9);
                let conflit = r == r2 || c == c2 || (r / 3 == r2 / 3 && c / 3 == c2 / 3);
                if conflit {
                    g.add_edge(NodeIndex::new(v), NodeIndex::new(w), ());
                }
            }
        }
        g
    }

    fn adjacence(g: &Graph<(), (), Undirected>) -> Vec<Vec<usize>> {
        (0..81usize)
            .map(|v| g.neighbors(NodeIndex::new(v)).map(|n| n.index()).collect())
            .collect()
    }

    fn parse_coloring(s: &str) -> Vec<usize> {
        s.chars()
            .filter(|ch| ch.is_ascii_digit() || *ch == '.')
            .map(|ch| if ch == '.' { 0 } else { ch as usize - '0' as usize })
            .collect()
    }

    fn couleurs_disponibles(v: usize, coloring: &[usize], adj: &[Vec<usize>]) -> Vec<usize> {
        let mut pris = [false; 10];
        for &n in &adj[v] {
            pris[coloring[n]] = true;
        }
        (1..=9usize).filter(|&d| !pris[d]).collect()
    }

    fn select_mrv(coloring: &[usize], adj: &[Vec<usize>]) -> Option<usize> {
        let mut meilleur: Option<usize> = None;
        let mut min_couleurs = usize::MAX;
        for v in 0..81usize {
            if coloring[v] != 0 {
                continue;
            }
            let n = couleurs_disponibles(v, coloring, adj).len();
            if n < min_couleurs {
                min_couleurs = n;
                meilleur = Some(v);
            }
        }
        meilleur
    }

    fn backtrack_mrv(
        coloring: &mut [usize],
        adj: &[Vec<usize>],
        noeuds: &mut usize,
        retours: &mut usize,
    ) -> bool {
        *noeuds += 1;
        let sommet = match select_mrv(coloring, adj) {
            None => return true,
            Some(v) => v,
        };
        for couleur in couleurs_disponibles(sommet, coloring, adj) {
            coloring[sommet] = couleur;
            if backtrack_mrv(coloring, adj, noeuds, retours) {
                return true;
            }
            coloring[sommet] = 0;
            *retours += 1;
        }
        false
    }

    fn resoudre_mrv(puzzle: &str, adj: &[Vec<usize>]) -> (bool, usize, usize, Vec<usize>) {
        let mut coloring = parse_coloring(puzzle);
        let mut noeuds = 0;
        let mut retours = 0;
        let ok = backtrack_mrv(&mut coloring, adj, &mut noeuds, &mut retours);
        (ok, noeuds, retours, coloring)
    }

    fn benchmark(nom: &str, puzzles: &[&str], adj: &[Vec<usize>]) {
        println!("\nBenchmark: {} ({} puzzles)", nom, puzzles.len());
        let mut succes = 0usize;
        let mut total_ms = 0.0f64;
        for (i, p) in puzzles.iter().enumerate() {
            let t0 = std::time::Instant::now();
            let (ok, noeuds, retours, _) = resoudre_mrv(p, adj);
            let ms = t0.elapsed().as_secs_f64() * 1000.0;
            println!(
                "  puzzle {} : succes={}, {:.2} ms, noeuds={}, backtracks={}",
                i, ok, ms, noeuds, retours
            );
            if ok {
                succes += 1;
                total_ms += ms;
            }
        }
        if succes > 0 {
            println!("Succes: {}/{}", succes, puzzles.len());
            println!("Temps moyen: {:.2} ms", total_ms / succes as f64);
        } else {
            println!("Aucun puzzle resolu !");
        }
    }

    // Puzzles faciles -- Puzzles/Sudoku_Easy51.txt, lignes 1-3 (0 = case vide)
    let faciles = [
        "902005403100063025508407060026309001057010290090670530240530600705200304080041950",
        "003020600900305001001806400008102900700000008006708200002609500800203009005010300",
        "200080300060070084030500209000105408000000000402706000301007040720040060004010003",
    ];
    // Puzzles difficiles -- Puzzles/Sudoku_hardest.txt, lignes 1-2 (. = case vide)
    let durs = [
        "85...24..72......9..4.........1.7..23.5...9...4...........8..7..17..........36.4.",
        "..53.....8......2..7..1.5..4....53...1..7...6..32...8..6.5....9..4....3......97..",
    ];

    let g = sudoku_graph();
    let adj = adjacence(&g);
    benchmark("Puzzles Faciles", &faciles, &adj);
    benchmark("Puzzles Difficiles", &durs, &adj);
};

Benchmark: Puzzles Faciles (3 puzzles)


  puzzle 0 : succes=true, 0.09 ms, noeuds=37, backtracks=0


  puzzle 1 : succes=true, 0.17 ms, noeuds=50, backtracks=0


  puzzle 2 : succes=true, 0.24 ms, noeuds=57, backtracks=5


Succes: 3/3


Temps moyen: 0.17 ms


Benchmark: Puzzles Difficiles (2 puzzles)


  puzzle 0 : succes=true, 17.69 ms, noeuds=4037, backtracks=3977


  puzzle 1 : succes=true, 1.77 ms, noeuds=543, backtracks=484


Succes: 2/2


Temps moyen: 9.73 ms


### Interprétation : benchmark coloration de graphe

| Type | Succès | Temps moyen | Analyse |
|------|--------|-------------|---------|
| **Puzzles faciles** | 3/3 | 0.17 ms | 37 à 57 noeuds, 0 à 5 backtracks : le domaine MRV guide presque directement à la solution |
| **Puzzles difficiles** | 2/2 | 9.73 ms | 543 puis 4037 noeuds : l'arbre s'élargit d'un ordre de grandeur sur hardest[0] |

**Points clés** :

1. **Le temps suit les backtracks**, pas la taille de la grille : la même exploration de 81 sommets passe de 0.24 ms (5 backtracks) à 17.69 ms (3977 backtracks).
2. **Les temps Rust** sont la résolution pure, à comparer au jumeau Python (~3.7 ms faciles, ~46 ms difficiles côté Python) : l'écart mesure à la fois le langage et la structure de données (`Vec` plate vs dictionnaires Python).
3. Les comptages `noeuds`/`backtracks` des puzzles difficiles diffèrent du jumeau Python (ordre d'itération des couleurs — voir la note de la cellule précédente) : `hardest[0]` passe de 164/104 côté Python à 4037/3977 ici, la première branche explorée n'étant pas la même.

### LCV — Least Constraining Value

**LCV** ordonne les **couleurs** plutôt que les sommets : au lieu d'essayer les couleurs par ordre croissant, on essaie d'abord celle qui **laisse le plus d'options aux voisins** — la moins contraignante. Pour chaque couleur candidate `c` du sommet choisi (par MRV, inchangé), on compte le total des couleurs encore disponibles pour les voisins non coloriés si `c` était assignée, puis on trie les candidates par ce compte décroissant.

Le coût est réel : chaque niveau de l'arbre paie un tri des candidates et un recomptage de domaines voisins. Sur un graphe régulier et fortement contraint comme le Sudoku, le gain n'est pas garanti — c'est exactement ce que la cellule suivante mesure sur le puzzle difficile `hardest[0]`, en comparant LCV au MRV pur (couleurs par ordre croissant).

In [7]:
// LCV vs MRV pur sur le puzzle difficile hardest[0]
{
    use petgraph::graph::NodeIndex;
    use petgraph::{Graph, Undirected};

    fn sudoku_graph() -> Graph<(), (), Undirected> {
        let mut g: Graph<(), (), Undirected> = Graph::with_capacity(81, 810);
        for _ in 0..81 {
            g.add_node(());
        }
        for v in 0..81usize {
            let (r, c) = (v / 9, v % 9);
            for w in (v + 1)..81usize {
                let (r2, c2) = (w / 9, w % 9);
                let conflit = r == r2 || c == c2 || (r / 3 == r2 / 3 && c / 3 == c2 / 3);
                if conflit {
                    g.add_edge(NodeIndex::new(v), NodeIndex::new(w), ());
                }
            }
        }
        g
    }

    fn adjacence(g: &Graph<(), (), Undirected>) -> Vec<Vec<usize>> {
        (0..81usize)
            .map(|v| g.neighbors(NodeIndex::new(v)).map(|n| n.index()).collect())
            .collect()
    }

    fn parse_coloring(s: &str) -> Vec<usize> {
        s.chars()
            .filter(|ch| ch.is_ascii_digit() || *ch == '.')
            .map(|ch| if ch == '.' { 0 } else { ch as usize - '0' as usize })
            .collect()
    }

    fn couleurs_disponibles(v: usize, coloring: &[usize], adj: &[Vec<usize>]) -> Vec<usize> {
        let mut pris = [false; 10];
        for &n in &adj[v] {
            pris[coloring[n]] = true;
        }
        (1..=9usize).filter(|&d| !pris[d]).collect()
    }

    fn select_mrv(coloring: &[usize], adj: &[Vec<usize>]) -> Option<usize> {
        let mut meilleur: Option<usize> = None;
        let mut min_couleurs = usize::MAX;
        for v in 0..81usize {
            if coloring[v] != 0 {
                continue;
            }
            let n = couleurs_disponibles(v, coloring, adj).len();
            if n < min_couleurs {
                min_couleurs = n;
                meilleur = Some(v);
            }
        }
        meilleur
    }

    fn count_remaining_options(
        sommet: usize,
        couleur: usize,
        coloring: &mut [usize],
        adj: &[Vec<usize>],
    ) -> usize {
        // Total des options restantes des voisins non colories si `couleur`
        // est assignee a `sommet` : plus le compte est eleve, moins la couleur
        // est contraignante (LCV).
        coloring[sommet] = couleur;
        let mut total = 0usize;
        for &n in &adj[sommet] {
            if coloring[n] == 0 {
                total += couleurs_disponibles(n, coloring, adj).len();
            }
        }
        coloring[sommet] = 0;
        total
    }

    fn backtrack_lcv(
        coloring: &mut [usize],
        adj: &[Vec<usize>],
        noeuds: &mut usize,
        retours: &mut usize,
    ) -> bool {
        *noeuds += 1;
        let sommet = match select_mrv(coloring, adj) {
            None => return true,
            Some(v) => v,
        };
        let mut candidates: Vec<(usize, usize)> = couleurs_disponibles(sommet, coloring, adj)
            .into_iter()
            .map(|c| (count_remaining_options(sommet, c, coloring, adj), c))
            .collect();
        // Tri du tuple (options, couleur) par ordre decroissant : la couleur la
        // moins contraignante d'abord, egalite departagee par la couleur la plus
        // grande -- le tri inverse du jumeau Python.
        candidates.sort_by(|a, b| b.cmp(a));
        for (_, couleur) in candidates {
            coloring[sommet] = couleur;
            if backtrack_lcv(coloring, adj, noeuds, retours) {
                return true;
            }
            coloring[sommet] = 0;
            *retours += 1;
        }
        false
    }

    fn backtrack_mrv(
        coloring: &mut [usize],
        adj: &[Vec<usize>],
        noeuds: &mut usize,
        retours: &mut usize,
    ) -> bool {
        *noeuds += 1;
        let sommet = match select_mrv(coloring, adj) {
            None => return true,
            Some(v) => v,
        };
        for couleur in couleurs_disponibles(sommet, coloring, adj) {
            coloring[sommet] = couleur;
            if backtrack_mrv(coloring, adj, noeuds, retours) {
                return true;
            }
            coloring[sommet] = 0;
            *retours += 1;
        }
        false
    }

    fn resoudre(puzzle: &str, adj: &[Vec<usize>], lcv: bool) -> (bool, usize, usize) {
        let mut coloring = parse_coloring(puzzle);
        let mut noeuds = 0;
        let mut retours = 0;
        let ok = if lcv {
            backtrack_lcv(&mut coloring, adj, &mut noeuds, &mut retours)
        } else {
            backtrack_mrv(&mut coloring, adj, &mut noeuds, &mut retours)
        };
        (ok, noeuds, retours)
    }

    let g = sudoku_graph();
    let adj = adjacence(&g);
    // hardest[0] -- Puzzles/Sudoku_hardest.txt, ligne 1
    let dur0 = "85...24..72......9..4.........1.7..23.5...9...4...........8..7..17..........36.4.";
    let (_, noeuds_lcv, retours_lcv) = resoudre(dur0, &adj, true);
    let (_, noeuds_mrv, retours_mrv) = resoudre(dur0, &adj, false);
    println!("LCV - Noeuds: {}, Backtracks: {}", noeuds_lcv, retours_lcv);
    println!("MRV pur - Noeuds: {}, Backtracks: {}", noeuds_mrv, retours_mrv);
};

LCV - Noeuds: 182, Backtracks: 122


MRV pur - Noeuds: 4037, Backtracks: 3977


### Interprétation : LCV vs MRV pur

| Méthode | Noeuds explorés | Backtracks | Analyse |
|---------|-----------------|------------|---------|
| **LCV** | 182 | 122 | la couleur « la moins contraignante d'abord » coupe l'arbre tôt |
| **MRV pur** | 4037 | 3977 | l'ordre croissant brut des couleurs s'enfonce dans une mauvaise branche |

**Points clés** :

1. **Ici, LCV paie largement son tri** : 182 noeuds contre 4037 — plus de vingt fois moins d'exploration pour un surcoût de tri par noeud. Sur un puzzle difficile, fixer un bon ordre des valeurs vaut bien plus qu'il ne coûte.
2. **Le jumeau Python mesure l'inverse** (LCV 182/122 contre MRV pur 164/104), et ce n'est pas une erreur : le MRV pur du jumeau itère les couleurs dans un `set` CPython, dont l'ordre interne tombait par chance sur une bonne branche. Le comptage LCV, lui, est **identique au jumeau** (182/122) : le tri explicite des candidates écrase l'ordre de départ, et le portage le reproduit exactement.
3. **La leçon est la sensibilité à l'ordre** : dès qu'il y a des égalités de domaine, le comptage d'un backtracking dépend de l'ordre d'essai des couleurs. Une heuristique de valeur comme LCV a aussi pour effet de **fixer** cet ordre — sa reproductibilité cesse d'être un accident d'implémentation.

### Exemple guidé : l'heuristique Welsh-Powell

**Welsh-Powell** est un glouton classique de coloration : ordonner les sommets par **degré décroissant** — ici degré **dynamique**, le nombre de voisins non coloriés — colorier le premier de la liste avec la plus petite couleur disponible, et backtracker en cas d'impasse.

Sur un graphe **hétérogène**, colorier d'abord les sommets les plus contraignants réduit l'arbre. Mais le graphe Sudoku est **régulier de degré 20** : au départ, le tri ne départage rien, et le choix du premier sommet devient quasi arbitraire. La cellule suivante mesure ce contraste sur le même puzzle facile que le jumeau Python (`easy[0]`), en exécutant les deux solveurs côte à côte.

In [8]:
// Welsh-Powell (degre dynamique decroissant) vs MRV sur easy[0]
{
    use petgraph::graph::NodeIndex;
    use petgraph::{Graph, Undirected};

    fn sudoku_graph() -> Graph<(), (), Undirected> {
        let mut g: Graph<(), (), Undirected> = Graph::with_capacity(81, 810);
        for _ in 0..81 {
            g.add_node(());
        }
        for v in 0..81usize {
            let (r, c) = (v / 9, v % 9);
            for w in (v + 1)..81usize {
                let (r2, c2) = (w / 9, w % 9);
                let conflit = r == r2 || c == c2 || (r / 3 == r2 / 3 && c / 3 == c2 / 3);
                if conflit {
                    g.add_edge(NodeIndex::new(v), NodeIndex::new(w), ());
                }
            }
        }
        g
    }

    fn adjacence(g: &Graph<(), (), Undirected>) -> Vec<Vec<usize>> {
        (0..81usize)
            .map(|v| g.neighbors(NodeIndex::new(v)).map(|n| n.index()).collect())
            .collect()
    }

    fn parse_coloring(s: &str) -> Vec<usize> {
        s.chars()
            .filter(|ch| ch.is_ascii_digit() || *ch == '.')
            .map(|ch| if ch == '.' { 0 } else { ch as usize - '0' as usize })
            .collect()
    }

    fn couleurs_disponibles(v: usize, coloring: &[usize], adj: &[Vec<usize>]) -> Vec<usize> {
        let mut pris = [false; 10];
        for &n in &adj[v] {
            pris[coloring[n]] = true;
        }
        (1..=9usize).filter(|&d| !pris[d]).collect()
    }

    fn select_mrv(coloring: &[usize], adj: &[Vec<usize>]) -> Option<usize> {
        let mut meilleur: Option<usize> = None;
        let mut min_couleurs = usize::MAX;
        for v in 0..81usize {
            if coloring[v] != 0 {
                continue;
            }
            let n = couleurs_disponibles(v, coloring, adj).len();
            if n < min_couleurs {
                min_couleurs = n;
                meilleur = Some(v);
            }
        }
        meilleur
    }

    fn backtrack_mrv(
        coloring: &mut [usize],
        adj: &[Vec<usize>],
        noeuds: &mut usize,
        retours: &mut usize,
    ) -> bool {
        *noeuds += 1;
        let sommet = match select_mrv(coloring, adj) {
            None => return true,
            Some(v) => v,
        };
        for couleur in couleurs_disponibles(sommet, coloring, adj) {
            coloring[sommet] = couleur;
            if backtrack_mrv(coloring, adj, noeuds, retours) {
                return true;
            }
            coloring[sommet] = 0;
            *retours += 1;
        }
        false
    }

    fn select_welsh_powell(coloring: &[usize], adj: &[Vec<usize>]) -> Option<usize> {
        // Sommet non colorie au degre dynamique maximal (voisins non colories) ;
        // premiere occurrence en cas d'egalite -- l'equivalent du tri stable
        // par degre decroissant du jumeau Python.
        let mut meilleur: Option<usize> = None;
        let mut max_deg = 0usize;
        for v in 0..81usize {
            if coloring[v] != 0 {
                continue;
            }
            let deg = adj[v].iter().filter(|&&n| coloring[n] == 0).count();
            if meilleur.is_none() || deg > max_deg {
                max_deg = deg;
                meilleur = Some(v);
            }
        }
        meilleur
    }

    fn backtrack_wp(
        coloring: &mut [usize],
        adj: &[Vec<usize>],
        noeuds: &mut usize,
        retours: &mut usize,
    ) -> bool {
        *noeuds += 1;
        let sommet = match select_welsh_powell(coloring, adj) {
            None => return true,
            Some(v) => v,
        };
        for couleur in couleurs_disponibles(sommet, coloring, adj) {
            coloring[sommet] = couleur;
            if backtrack_wp(coloring, adj, noeuds, retours) {
                return true;
            }
            coloring[sommet] = 0;
            *retours += 1;
        }
        false
    }

    fn resoudre(puzzle: &str, adj: &[Vec<usize>], wp: bool) -> (bool, usize, usize) {
        let mut coloring = parse_coloring(puzzle);
        let mut noeuds = 0;
        let mut retours = 0;
        let ok = if wp {
            backtrack_wp(&mut coloring, adj, &mut noeuds, &mut retours)
        } else {
            backtrack_mrv(&mut coloring, adj, &mut noeuds, &mut retours)
        };
        (ok, noeuds, retours)
    }

    let g = sudoku_graph();
    let adj = adjacence(&g);
    // easy[0] -- Puzzles/Sudoku_Easy51.txt, ligne 1
    let facile0 = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";
    let (succes_wp, noeuds_wp, retours_wp) = resoudre(facile0, &adj, true);
    let (succes_mrv, noeuds_mrv, retours_mrv) = resoudre(facile0, &adj, false);
    println!(
        "Welsh-Powell (easy[0]) - Succes: {}, Noeuds: {}, Backtracks: {}",
        succes_wp, noeuds_wp, retours_wp
    );
    println!(
        "MRV (easy[0]) - Succes: {}, Noeuds: {}, Backtracks: {}",
        succes_mrv, noeuds_mrv, retours_mrv
    );
};

Welsh-Powell (easy[0]) - Succes: true, Noeuds: 992, Backtracks: 955


MRV (easy[0]) - Succes: true, Noeuds: 37, Backtracks: 0


### Interprétation : Welsh-Powell vs MRV sur Sudoku

| Méthode | Noeuds explorés | Backtracks | Analyse |
|---------|-----------------|------------|---------|
| **Welsh-Powell** | 992 | 955 | presque chaque assignation est remontée |
| **MRV pur** | 37 | 0 | résolution sans erreur |

**Points clés** :

1. **Welsh-Powell est correct mais sous-optimal** : il produit une solution valide en 992 noeuds — 27 fois plus que MRV — avec 955 retours arrière sur la même instance où MRV n'en fait aucun.
2. **Graphe régulier = heuristique inopérante** : tous les sommets non coloriés ont un degré dynamique similaire, le tri par degré n'apporte pas de guide discriminant — le choix du sommet redevient quasi arbitraire.
3. **MRV exploite la propagation** : choisir la case au plus petit domaine contraint mécaniquement la recherche.
4. **Reproductibilité du portage** : 992/955 sont exactement les comptages du jumeau Python sur la même instance — le tri stable par degré (première occurrence à égalité) et les couleurs par ordre croissant se correspondent terme à terme.
5. **Généralisation** : Welsh-Powell reste pertinent sur des graphes à degrés hétérogènes (allocation de registres, coloration de cartes) ; pour le Sudoku fortement contraint et régulier, MRV et ses variantes sont plus adaptées.

## 4. Exercices

### Exercice 1 — Compter les contraintes par cellule

**Objectif** : implémenter `count_unassigned_constraints(coloring, adj)` qui retourne, pour chaque cellule **non coloriée**, combien de ses voisins sont également non coloriés — les cellules les plus contraintes sont les plus difficiles à remplir.

**Indice** :
- Étape 1 : identifier les sommets non coloriés (`coloring[v] == 0`)
- Étape 2 : pour chaque sommet non colorié, compter ses voisins non coloriés (`adj[v]`)
- Étape 3 : retourner les couples `(sommet, compte)` triés par compte **décroissant** (à égalité, sommet croissant)

La valeur de retour attendue est un `Vec<(usize, usize)>` — l'équivalent du dictionnaire trié du jumeau Python.

In [9]:
// EXERCICE 1 : Compter les contraintes par cellule
{
    use petgraph::graph::NodeIndex;
    use petgraph::{Graph, Undirected};

    fn sudoku_graph() -> Graph<(), (), Undirected> {
        let mut g: Graph<(), (), Undirected> = Graph::with_capacity(81, 810);
        for _ in 0..81 {
            g.add_node(());
        }
        for v in 0..81usize {
            let (r, c) = (v / 9, v % 9);
            for w in (v + 1)..81usize {
                let (r2, c2) = (w / 9, w % 9);
                let conflit = r == r2 || c == c2 || (r / 3 == r2 / 3 && c / 3 == c2 / 3);
                if conflit {
                    g.add_edge(NodeIndex::new(v), NodeIndex::new(w), ());
                }
            }
        }
        g
    }

    fn adjacence(g: &Graph<(), (), Undirected>) -> Vec<Vec<usize>> {
        (0..81usize)
            .map(|v| g.neighbors(NodeIndex::new(v)).map(|n| n.index()).collect())
            .collect()
    }

    fn count_unassigned_constraints(coloring: &[usize], adj: &[Vec<usize>]) -> Vec<(usize, usize)> {
        // Pour chaque cellule non coloriee, combien de voisins non colories.
        // TODO etudiant : implementez le comptage des contraintes
        let _ = (coloring, adj);
        Vec::new()
    }

    let g = sudoku_graph();
    let adj = adjacence(&g);
    // Grille partiellement remplie (ligne 0 : 4,_,3,_,2,_,6,_,7)
    let mut test_coloring = vec![0usize; 81];
    for (i, v) in [4, 0, 3, 0, 2, 0, 6, 0, 7].iter().enumerate() {
        test_coloring[i] = *v;
    }
    let constraints = count_unassigned_constraints(&test_coloring, &adj);
    println!("Nombre de cellules non coloriees : {}", constraints.len());
    if !constraints.is_empty() {
        let top: Vec<(usize, usize)> = constraints.iter().take(3).copied().collect();
        println!("Top 3 cellules les plus contraintes : {:?}", top);
    }
};

Nombre de cellules non coloriees : 0


### Exercice 2 — Heuristique de degré (MRV + degré)

**Objectif** : implémenter `select_unassigned_vertex(coloring, adj)` qui combine MRV et heuristique de degré. Quand plusieurs sommets ont le même nombre de couleurs disponibles (égalité MRV), l'**heuristique de degré** les départage en choisissant celui qui a le **plus de voisins non coloriés** — le plus contraignant, à colorier en premier pour réduire l'arbre.

**Indice** :
- Étape 1 : collecter les sommets non coloriés (`coloring[v] == 0`)
- Étape 2 : pour chacun, calculer le nombre de couleurs disponibles (1..=9 non prises par les voisins)
- Étape 3 : trouver le minimum (MRV)
- Étape 4 : en cas d'égalité, départager avec le degré dynamique (nombre de voisins non coloriés) ; à égalité de degré, prendre le plus petit indice

La valeur de retour est un `Option<usize>` — le portage Rust du `-1` du jumeau Python : `None` signifie « tous les sommets sont coloriés ».

In [10]:
// EXERCICE 2 : Heuristique de degre (MRV + degre)
{
    use petgraph::graph::NodeIndex;
    use petgraph::{Graph, Undirected};

    fn sudoku_graph() -> Graph<(), (), Undirected> {
        let mut g: Graph<(), (), Undirected> = Graph::with_capacity(81, 810);
        for _ in 0..81 {
            g.add_node(());
        }
        for v in 0..81usize {
            let (r, c) = (v / 9, v % 9);
            for w in (v + 1)..81usize {
                let (r2, c2) = (w / 9, w % 9);
                let conflit = r == r2 || c == c2 || (r / 3 == r2 / 3 && c / 3 == c2 / 3);
                if conflit {
                    g.add_edge(NodeIndex::new(v), NodeIndex::new(w), ());
                }
            }
        }
        g
    }

    fn adjacence(g: &Graph<(), (), Undirected>) -> Vec<Vec<usize>> {
        (0..81usize)
            .map(|v| g.neighbors(NodeIndex::new(v)).map(|n| n.index()).collect())
            .collect()
    }

    fn select_unassigned_vertex(coloring: &[usize], adj: &[Vec<usize>]) -> Option<usize> {
        // TODO etudiant : implementez MRV + heuristique de degre
        let _ = (coloring, adj);
        None
    }

    let g = sudoku_graph();
    let adj = adjacence(&g);
    // Grille partiellement remplie (ligne 0 : 4,_,3,_,2,_,6,_,7)
    let mut test_coloring = vec![0usize; 81];
    for (i, v) in [4, 0, 3, 0, 2, 0, 6, 0, 7].iter().enumerate() {
        test_coloring[i] = *v;
    }
    let sommet = select_unassigned_vertex(&test_coloring, &adj);
    println!("Prochain sommet a colorier : {:?}", sommet);
};

Prochain sommet a colorier : None


### Exercice 3 — Vérification d'une coloration valide

**Objectif** : implémenter `is_valid_coloring(coloring, adj)` qui vérifie qu'une coloration est **propre et complète** : deux sommets adjacents ne partagent jamais la même couleur, et aucun sommet ne reste à `0`.

**Indice** :
- Étape 1 : vérifier que chaque sommet est colorié (`coloring[v] != 0`)
- Étape 2 : pour chaque arete `(v, w)` de `adj`, vérifier `coloring[v] != coloring[w]`
- Étape 3 : retourner `true` si les deux conditions tiennent, `false` sinon

La cellule résout d'abord une grille avec le solveur MRV de la section 3 (redéfini ici — cellule autonome), puis soumet la coloration obtenue à votre vérification : c'est la **vérification indépendante** du solveur.

In [11]:
// EXERCICE 3 : Verification d'une coloration valide
{
    use petgraph::graph::NodeIndex;
    use petgraph::{Graph, Undirected};

    fn sudoku_graph() -> Graph<(), (), Undirected> {
        let mut g: Graph<(), (), Undirected> = Graph::with_capacity(81, 810);
        for _ in 0..81 {
            g.add_node(());
        }
        for v in 0..81usize {
            let (r, c) = (v / 9, v % 9);
            for w in (v + 1)..81usize {
                let (r2, c2) = (w / 9, w % 9);
                let conflit = r == r2 || c == c2 || (r / 3 == r2 / 3 && c / 3 == c2 / 3);
                if conflit {
                    g.add_edge(NodeIndex::new(v), NodeIndex::new(w), ());
                }
            }
        }
        g
    }

    fn adjacence(g: &Graph<(), (), Undirected>) -> Vec<Vec<usize>> {
        (0..81usize)
            .map(|v| g.neighbors(NodeIndex::new(v)).map(|n| n.index()).collect())
            .collect()
    }

    fn parse_coloring(s: &str) -> Vec<usize> {
        s.chars()
            .filter(|ch| ch.is_ascii_digit() || *ch == '.')
            .map(|ch| if ch == '.' { 0 } else { ch as usize - '0' as usize })
            .collect()
    }

    fn couleurs_disponibles(v: usize, coloring: &[usize], adj: &[Vec<usize>]) -> Vec<usize> {
        let mut pris = [false; 10];
        for &n in &adj[v] {
            pris[coloring[n]] = true;
        }
        (1..=9usize).filter(|&d| !pris[d]).collect()
    }

    fn select_mrv(coloring: &[usize], adj: &[Vec<usize>]) -> Option<usize> {
        let mut meilleur: Option<usize> = None;
        let mut min_couleurs = usize::MAX;
        for v in 0..81usize {
            if coloring[v] != 0 {
                continue;
            }
            let n = couleurs_disponibles(v, coloring, adj).len();
            if n < min_couleurs {
                min_couleurs = n;
                meilleur = Some(v);
            }
        }
        meilleur
    }

    fn backtrack_mrv(
        coloring: &mut [usize],
        adj: &[Vec<usize>],
        noeuds: &mut usize,
        retours: &mut usize,
    ) -> bool {
        *noeuds += 1;
        let sommet = match select_mrv(coloring, adj) {
            None => return true,
            Some(v) => v,
        };
        for couleur in couleurs_disponibles(sommet, coloring, adj) {
            coloring[sommet] = couleur;
            if backtrack_mrv(coloring, adj, noeuds, retours) {
                return true;
            }
            coloring[sommet] = 0;
            *retours += 1;
        }
        false
    }

    fn is_valid_coloring(coloring: &[usize], adj: &[Vec<usize>]) -> bool {
        // TODO etudiant : implementez la verification de coloration
        let _ = (coloring, adj);
        false
    }

    let g = sudoku_graph();
    let adj = adjacence(&g);
    // Grille de test -- meme instance que le jumeau Python
    let puzzle_test = "..3.2.6..9..3.5..1..18.64....81.29..7.......8..67.82....26.95..8..2.3..9..5.1.3..";
    let mut coloring = parse_coloring(puzzle_test);
    let mut noeuds = 0;
    let mut retours = 0;
    let succes = backtrack_mrv(&mut coloring, &adj, &mut noeuds, &mut retours);
    if succes {
        println!("Coloration valide : {}", is_valid_coloring(&coloring, &adj));
    } else {
        println!("Pas de solution");
    }
};

Coloration valide : false


## À retenir

- **L'organe natif est la crate** : `petgraph` fournit la structure de graphe (`Graph`, `neighbors`, `node_count`, `edge_count`) ; contrairement à `nx.sudoku_graph()`, le **générateur** du graphe Sudoku reste à écrire — 81 sommets, 810 aretes, régulier de degré 20, mesuré tel quel en section 2.
- **Résoudre = colorer** : les 81 cases sont les sommets, deux cases en conflit (ligne, colonne, bloc) sont reliées, et une solution est une **coloration propre à 9 couleurs**.
- **L'heuristique de sommet fait tout** : sur le même puzzle facile, MRV résout en 37 noeuds / 0 backtracks quand Welsh-Powell en demande 992 / 955 — sur un graphe régulier, trier par degré ne départage rien.
- **L'ordre des valeurs décide aussi** : sur `hardest[0]`, LCV (ordre trié) explore 182 noeuds quand le MRV pur (ordre croissant brut) en explore 4037 — et le jumeau Python, qui itère un `set`, mesure l'inverse pour le MRV pur. Dès qu'il y a égalité de domaine, le comptage dépend de l'ordre d'essai : une heuristique de valeur comme LCV fixe aussi cet ordre.
- **Cellules autonomes** : sous evcxr, chaque cellule redéfinit ses fonctions et reconstruit son graphe — le coût de construction est négligeable, la robustesse maximale.
- **Ce que Rust apporte** : aucun gain de vitesse revendiqué ici — l'apport est la conformité mesurée : 4 grandeurs (81 sommets, 810 arêtes, Welsh-Powell 992 nœuds, MRV 37 nœuds sur easy[0]) sont identiques aux sorties du jumeau Python, tolérance nulle (registre `rust_conformance.d/sudoku-09-graphcoloring.yaml`).


## Vérifiez votre compréhension

**1. Pourquoi le graphe du Sudoku est-il régulier de degré 20 ?**

<details><summary>Réponse</summary>

Chaque case est en conflit avec les cases de sa ligne (8), de sa colonne (8) et de son bloc (4, hors ligne et colonne déjà comptées) : **8 + 8 + 4 = 20 voisins**, identiques pour chaque case. La mesure `petgraph` de la section 2 confirme `Degré min: 20, max: 20` sur les 81 sommets et 810 aretes.

</details>

**2. Que mesure la comparaison Welsh-Powell vs MRV sur le même puzzle ?**

<details><summary>Réponse</summary>

Sur `easy[0]`, Welsh-Powell explore 992 noeuds / 955 backtracks, MRV 37 / 0 — vingt-sept fois moins. Les deux résolvent, mais MRV (choisir la case au plus petit domaine restant) évite quasi tout faux pas. Leçon : **l'heuristique d'ordonnancement fait tout** dans un problème à contraintes — et sur un graphe régulier comme celui du Sudoku, trier par degré n'apporte pas de guide.

</details>

**3. Qu'apporte `petgraph` par rapport à `nx.sudoku_graph()`, et qu'apporte-t-il de moins ?**

<details><summary>Réponse</summary>

`petgraph` apporte la **structure** : conteneur de graphe typé (`Graph<(), (), Undirected>`), itération de voisinage (`neighbors(NodeIndex)`), statistiques (`node_count`, `edge_count`), et toute une bibliothèque d'algorithmes (plus courts chemins, connectivité…). Il n'apporte **pas le générateur** : `nx.sudoku_graph()` produit le graphe du Sudoku en un appel, `petgraph` laisse les 810 aretes à construire — une double boucle `w > v` sur ligne/colonne/bloc, dédoublonnée par construction.

</details>

## Pour aller plus loin

- **Jumeaux du même carnet** : [Sudoku-09-GraphColoring-Python.ipynb](Sudoku-09-GraphColoring-Python.ipynb) (networkx et son générateur `sudoku_graph`) et [Sudoku-09-GraphColoring-CSharp.ipynb](Sudoku-09-GraphColoring-CSharp.ipynb) (QuikGraph côté .NET) — les mêmes heuristiques MRV, LCV et Welsh-Powell dans trois écosystèmes, à trois allures différentes.
- **Passer à l'organe industriel** : [Sudoku-10-ORTools-Python.ipynb](Sudoku-10-ORTools-Python.ipynb) reformule le Sudoku en programmation par contraintes (CP-SAT) et confronte le solveur au backtracking écrit à la main.
- **La crate** : documentation de [petgraph](https://docs.rs/petgraph) — parcours, plus courts chemins, connectivité, dominators : au-delà du conteneur utilisé ici, une boîte à outils de théorie des graphes.
- **Le générateur absent** : la référence [`networkx.generators.sudoku.sudoku_graph`](https://networkx.org/documentation/stable/reference/generated/networkx.generators.sudoku.sudoku_graph.html) que ce carnet réécrit à la main — 810 aretes issues d'une double boucle `w > v` sur ligne, colonne et bloc.